# HW3: Campus Edge

**COMPSS 211A | Fall 2026 | Student copy**

In this homework you work with real 311 service requests from the streets around campus. You get the same records in four file formats. You will load all four, check that they really contain the same records, and make one small summary table.

**Due:** Sunday, October 25, 2026 at 11:59 p.m.

Fill in every cell marked **Your turn** and answer the written questions. You won't just run code: you'll change it, extend it, and sometimes predict what it does before you run it. When a cell asks for a prediction, write your guess **first**. Guessing wrong is fine; that's often where you learn the most.

Keep the variable names we ask for, because we look for them when grading.

## Scenario

311 is Berkeley's non-emergency service line. People use it to report things like illegal dumping, noise, or a tree that needs pruning. Each report becomes a *case* with its own ID number.

The fictional Berkeley Methods Studio wants a short note on what kinds of requests people make near campus. We already downloaded the data from the City of Berkeley's open data website, so you do not need an internet connection or an API key.

## What you will practice

- Reading an API request and saying in plain words what it asks for (Week 6).
- Loading CSV, TSV, JSON, and XML files into pandas (Week 5 and Lab 5).
- Checking that files which should match actually do.
- Counting categories and turning counts into shares.
- Saying what a dataset can and cannot tell you.

## Helpful references

- **Lab 5** loads these exact four files. Look back at it whenever you get stuck in Tasks 2 and 3.
- **Week 6 API reference** (`lessons/week06_web-apis/01_web_api_reference.md`) explains requests, parameters, and paging.
- **IDs are text, not numbers.** Load `case_id` as text (`dtype={"case_id": str}`). An ID is a label, like a phone number: you never add IDs together, and storing them as numbers can quietly change them.

## AI and collaboration policy

You may use AI to help you understand an error message, as long as you say so in a note in your notebook. Write the code and the written answers yourself. Talking through ideas with classmates is fine; share ideas, not code.

In [ ]:
from pathlib import Path
import json
import os
import sys

import pandas as pd
import requests
from xml.etree import ElementTree as ET
from IPython.display import display

SUPPORTED_PYTHON = (3, 13)
if sys.version_info[:2] != SUPPORTED_PYTHON:
    print(
        "Setup note: this course is tested with Python 3.13; "
        f"you are running {sys.version.split()[0]}."
    )

def find_course_root():
    """Find the cloned repository when this notebook is running locally."""
    for folder in (Path.cwd(), *Path.cwd().parents):
        if (folder / "pyproject.toml").exists() and (folder / "data").is_dir():
            return folder
    return None

LOCAL_COURSE_ROOT = find_course_root()
COURSE_ROOT = LOCAL_COURSE_ROOT or Path.cwd()
DATA_DIR = (
    LOCAL_COURSE_ROOT / "data"
    if LOCAL_COURSE_ROOT
    else COURSE_ROOT / "compss211_data"
)
GENERATED_DIR = COURSE_ROOT / "generated"
DATA_DIR.mkdir(parents=True, exist_ok=True)
GENERATED_DIR.mkdir(parents=True, exist_ok=True)

DATA_BASE_URL = (
    "https://raw.githubusercontent.com/"
    "macss-berkeley/compss-211a/main/data"
)

def course_data_path(filename):
    """Use a local course file, or download it when running in Colab."""
    path = DATA_DIR / filename
    if not path.exists():
        from urllib.request import urlretrieve

        urlretrieve(f"{DATA_BASE_URL}/{filename}", path)
    return path

print(f"Python {sys.version.split()[0]} | data={DATA_DIR}")

## 1. Read the API request

The data came from the City of Berkeley's API. An API request is a web address plus a set of *parameters*: settings that tell the server which records you want. In Python we store the parameters in a dictionary.

The cell below shows the exact request we used. You don't need to run it against the internet. Read it and the comments, then change it in the next cell.

One detail to know: this API sends back at most 1,000 records per request. Our data has 1,896 records, so we had to ask twice. The `$offset` parameter says how many records to skip. The first request uses `$offset` 0, and the second uses 1000 to skip the records we already have. This is called *paging*.

In [ ]:
API_ENDPOINT = "https://data.cityofberkeley.info/resource/p88g-6gs2.json"

query_params = {
    # Which columns to send back.
    "$select": "case_id, date_opened, case_status, last_action_date, date_closed, "
               "case_request, city, state, latitude, longitude",
    # Which rows to keep: cases opened in 2025, inside a box around campus.
    "$where": (
        "date_opened >= '2025-01-01' AND date_opened < '2026-01-01' "
        "AND latitude BETWEEN 37.862 AND 37.884 "
        "AND longitude BETWEEN -122.274 AND -122.244"
    ),
    # Sort by case ID so that every page comes back in the same order.
    "$order": "case_id",
    # At most 1,000 records per request.
    "$limit": 1000,
    # How many records to skip before starting.
    "$offset": 0,
}

# This is how the request would be sent. We don't call it in this homework.
def fetch_page(params):
    response = requests.get(API_ENDPOINT, params=params, timeout=30)
    response.raise_for_status()
    return response.json()

query_params

In [ ]:
# Your turn: make the parameters for the SECOND page of results.
# 1. Make a copy of query_params and call it second_page_params.
#    (Use .copy() so you don't change the original dictionary.)
# 2. Change "$offset" in the copy so it skips the first 1,000 records.
# 3. Display second_page_params.


# Your turn: now suppose the Studio wants the same area for 2024 instead of 2025.
# 1. Make another copy of query_params called params_2024.
# 2. Write a new "$where" string for it. Only the dates should change.
# 3. Display params_2024.

### Your response


Answer in two or three sentences, in your own words:

1. Which records does `$where` keep, and which does it leave out?
2. Why did we need a second request with a different `$offset`?
3. If `$limit` were 500 instead of 1000, how many requests would it take to get all 1,896 records, and what `$offset` would each one use?


> Write your response here, then delete this line.

## 2. Load all four files

The folder `data/` has the same records saved four ways:

| Format | File | How to load it |
| --- | --- | --- |
| CSV | `hw3_campus_edge_311_2025.csv` | `pd.read_csv(path, dtype={"case_id": str})` |
| TSV | `hw3_campus_edge_311_2025.tsv` | like CSV, but add `sep="\t"` because the columns are separated by tabs |
| JSON | `hw3_campus_edge_311_2025.json` | `pd.read_json(path, dtype={"case_id": str})` |
| XML | `hw3_campus_edge_311_2025.xml` | we give you this one below |

Create `csv_rows`, `tsv_rows`, and `json_rows`. The XML code is done for you. Then put all four tables into one dictionary called `raw_snapshots`.

In [ ]:
snapshot_paths = {
    "csv": course_data_path("hw3_campus_edge_311_2025.csv"),
    "tsv": course_data_path("hw3_campus_edge_311_2025.tsv"),
    "json": course_data_path("hw3_campus_edge_311_2025.json"),
    "xml": course_data_path("hw3_campus_edge_311_2025.xml"),
}

# XML (same code as Lab 5): every <service_request> becomes one row,
# and each tag inside it becomes a column.
xml_root = ET.parse(snapshot_paths["xml"]).getroot()
xml_rows = pd.DataFrame([{child.tag: child.text for child in item} for item in xml_root])
xml_rows.head()

In [ ]:
# Your turn: load the CSV, TSV, and JSON files.
csv_rows = ...
tsv_rows = ...
json_rows = ...

# Then put all four tables in one dictionary.
raw_snapshots = {
    "csv": csv_rows,
    # add tsv, json, and xml here
}

for file_format, table in raw_snapshots.items():
    print(file_format, table.shape)

Same rows, same columns. But are they the same *types*? In XML, every value is just text sitting between two tags, like `<latitude>37.86</latitude>`. Nothing in the file says "this one is a number".

**Predict first:** which columns do you think are numbers in `csv_rows` but text in `xml_rows`? Write your guess as a comment. Then check by displaying `csv_rows.dtypes` and `xml_rows.dtypes` side by side (hint: `pd.DataFrame({"csv": ..., "xml": ...})`). In a dtypes list, text shows up as `object`.

Then fix it: turn `latitude` and `longitude` in `xml_rows` into numbers with `pd.to_numeric(...)`, and show that the average latitude is now the same in `csv_rows` and `xml_rows`. (Before the fix, `.mean()` on the XML column would fail. You can try it to see the error.)

In [ ]:
# My prediction:
#

# Your turn: compare the dtypes, then fix latitude and longitude in xml_rows.

## 3. Check that the files match

Four files that *should* hold the same records don't always do. A row can get lost, an ID can change, and a blank value can be saved differently in each format. Before you analyze anything, check.

Build a table called `reconciliation` with one row per format and these columns:

- `format`: the name of the format
- `rows`: how many rows the table has
- `unique_cases`: how many different `case_id` values it has
- `same_ids_as_csv`: `True` if its set of case IDs is exactly the same as the CSV's
- `missing_date_closed`: how many values in `date_closed` pandas counts as missing (`.isna().sum()`)

The CSV row is done for you. Copy it and change it for the other three formats.

In [ ]:
csv_ids = set(csv_rows["case_id"])

reconciliation = pd.DataFrame([
    {
        "format": "csv",
        "rows": len(csv_rows),
        "unique_cases": csv_rows["case_id"].nunique(),
        "same_ids_as_csv": set(csv_rows["case_id"]) == csv_ids,
        "missing_date_closed": csv_rows["date_closed"].isna().sum(),
    },
    # Your turn: add one dictionary like the one above for tsv, json, and xml.
])
reconciliation

Look at the `missing_date_closed` column. One format disagrees. Find out why: display the first few `date_closed` values in that format next to the CSV's. A case that is still open has no closing date. How does each format write "no date"?

**Your turn:** fix the format that disagrees so pandas counts its blank dates as missing. Hint: `.replace("", pd.NA)` turns empty strings into pandas' missing value. Then print its missing count again. It should now match the CSV.

Last, make the table you'll analyze. We'll use the CSV version.

1. Make a copy of `csv_rows` called `clean_api_data`.
2. Turn `date_opened` into real dates with `pd.to_datetime(...)`.
3. Save it as `generated/hw3_campus_edge_requests.csv` without the index (`index=False`). `GENERATED_DIR` already points to the `generated/` folder.

In [ ]:
# Your turn: look at date_closed in the format that disagrees, then fix it.


# Your turn: make, fix, and save clean_api_data.
clean_api_data = ...

## 4. Count the request types

Which kinds of requests come up most often near campus? Write a small function, `summarize_request_types(frame, n=8)`, that returns a table with the `n` most common values of `case_request` and these columns:

- `case_request`: the request type
- `requests`: how many cases have that type
- `share`: that count divided by the number of rows in the **whole** table (not just the top `n`)

Hint: `frame["case_request"].value_counts()` gives you the counts, already sorted from most to least common. Then call your function on `clean_api_data` and save the result as `request_summary`.

The point of writing a function is that you can reuse it. Once it works, call it again on **only the cases that are still open** (`case_status` equal to `"Open"`), with `n=5`. Save that as `open_summary`.

In [ ]:
def summarize_request_types(frame, n=8):
    """Return the n most common request types with their counts and shares."""
    # Your turn: replace the line below with your code.
    raise NotImplementedError("Write summarize_request_types")


request_summary = summarize_request_types(clean_api_data)
display(request_summary)

# Your turn: open_summary

### Your response


Answer in two or three sentences: in `open_summary`, what number is `share` divided by? Give one reason why "share of open cases" could tell a different story from "share of all cases".


> Write your response here, then delete this line.

## 5. Write a short methods note

Write four to six sentences for the Berkeley Methods Studio. Include:

1. **Do the files match?** Point to at least one number from `reconciliation`.
2. **One difference between formats.** What did you notice about `date_closed` (or anything else), and why does it matter if you only looked at one format?
3. **What the summary shows.** Name the most common request type and its share.
4. **One limit.** What can 311 data *not* tell you? For example: a report shows that someone chose to call the city, not how often a problem actually happens.

### Your response

**Question:** What evidence shows the four files agree, and what are the limits of this data?

> Write your response here, then delete this line.